# Application Latency and Cost: Diagnose Before You Tune

> In 2010, Benjamin Sigelman and colleagues published Google's Dapper paper, showing why one request crossing many services needs a correlated trace before an operator can explain latency. OpenTelemetry later standardized the spans and metrics used for that job. Riverside House now needs the same view across gateway, cache, embedding, retrieval, reranking, generation, and retries.
>
> **Where you are:** The gateway chapter gave you routing, retries, caching, and cost controls. Inference systems separated prefill, decode, TTFT, TPOT, batching, and KV-cache behavior. Profiling established that measurement comes before optimization. This notebook supplies the missing application ledger.
>
> **Notation:** $L_r$ is request latency; $L_{r,s,a}$ is stage-attempt latency; $p_q(X)$ is nearest-rank percentile $q$; $T_{first}$ is TTFT; $T_{token}$ is TPOT; $C_r$ is request cost; $A_g/R_g$ is generation-attempt amplification.

> **Input:** frozen shared request traces. **Output:** retained `artifacts/latency-cost-report.json`. **Evidence state:** local fixture workflow executed successfully in the unified FDE environment; notebook outputs were cleared. Production and cloud behavior remain `UNVALIDATED`.

Concept owners: [Gateway](../../../genai/06-llm-gateway/06-llm-gateway.ipynb), [Inference Systems](../../../ai-infrastructure/07-inference-systems/inference-systems.ipynb), [Profiling](../../../ai-infrastructure/03-profiling/pytorch-profiling.ipynb), [telemetry contract](../../../../projects/riverside-ai-platform/contracts/v1/telemetry-attributes.schema.json), and [cost/capacity assumptions](../../../../projects/riverside-ai-platform/docs/cost-and-capacity-assumptions.md).

## 0 · The Challenge and Scope

> **The mission**: Riverside House release `rel-riv-002` - reconcile 100% of latency and cost, explain 80% success, and identify the first bottleneck without inventing a production SLO.

Five traces report 201.4 ms average latency. **But** they also contain a 7 ms cache hit, 285 ms tail requests, retries, and one paid failure. The average has no owner.

| Coverage | Topics |
|---|---|
| Built and measured | Conservation, stage p50/p95, TTFT/TPOT, retries, cache, useful throughput, token/cost normalization |
| Explained and linked | Queue/prefill/decode and GPU profiling |
| Named with production checks | Concurrency, autoscaling, provider billing, Azure |

```mermaid
flowchart LR
 A["Average"] --> B["Failure: tail and owner hidden"] --> C["Stage ledger and p95"] --> D["TTFT, retries, cache"] --> E["Useful cost and next test"]
 classDef input fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 classDef process fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 classDef failure fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 classDef success fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 class A input
 class B failure
 class C,D process
 class E success
```

## 1 · Reconcile the Ledger

Enforce $L_r=\sum_{s,a}L_{r,s,a}$ and $C_r=\sum_{s,a}C_{r,s,a}$ before aggregating. This fixture is explicitly serial; parallel production spans require critical-path analysis.

```mermaid
flowchart LR
 A["Frozen JSONL"] --> B["Schema"] --> C["Request and stage tables"] --> D{"Totals reconcile?"}
 D -->|No| E["Block analysis"]
 D -->|Yes| F["Permit attribution"]
 classDef input fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 classDef process fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 classDef caution fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 classDef failure fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 classDef success fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 class A input
 class B,C process
 class D caution
 class E failure
 class F success
```

In [ ]:
# -- Load, validate, flatten, and reconcile -------------------------------
from __future__ import annotations
from datetime import datetime, timezone
import hashlib, json, math
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from jsonschema import Draft202012Validator, FormatChecker

def repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / 'AUTHORING_GUIDE.md').is_file() and (candidate / 'learning/role-based-tracks/ai-engineer/shared').is_dir():
            return candidate
    searched = ' -> '.join(str(candidate) for candidate in (start, *start.parents))
    raise FileNotFoundError(
        f'Could not locate the ai-portfolio repository root from {start}. '
        'Expected both AUTHORING_GUIDE.md and learning/role-based-tracks/ai-engineer/shared in one ancestor. '
        f'Searched: {searched}'
    )

ROOT = repo_root(Path.cwd().resolve())
CHAPTER = ROOT/'learning/role-based-tracks/ai-engineer/03-application-latency-and-cost'
SHARED = ROOT/'learning/role-based-tracks/ai-engineer/shared'
FIXTURES = SHARED/'latency-cost'
TRACE = FIXTURES/'request-traces.jsonl'
fixture_version = (SHARED/'VERSION').read_text(encoding='utf-8').strip()
fixture_manifest = json.loads((SHARED/'fixture-manifest.json').read_text(encoding='utf-8'))
if fixture_manifest['fixture_version'] != fixture_version:
    raise RuntimeError('Fixture VERSION and fixture-manifest.json disagree')
for relative_path in (
    'latency-cost/request-traces.jsonl',
    'latency-cost/request-trace.schema.json',
    'latency-cost/EXPECTED_OUTCOMES.md',
):
    expected_digest = fixture_manifest['files'].get(relative_path)
    if expected_digest is None:
        raise RuntimeError(f'Fixture manifest does not pin {relative_path}')
    actual_digest = hashlib.sha256((SHARED/relative_path).read_bytes()).hexdigest()
    if actual_digest != expected_digest:
        raise RuntimeError(
            f'Stale or modified fixture: {relative_path}. '
            'Restore the pinned bytes or intentionally version the shared fixture contract.'
        )
schema = json.loads((FIXTURES/'request-trace.schema.json').read_text(encoding='utf-8'))
records = [json.loads(line) for line in TRACE.read_text(encoding='utf-8').splitlines() if line.strip()]
validator = Draft202012Validator(schema, format_checker=FormatChecker())
errors = [(r['request_id'],e.message) for r in records for e in validator.iter_errors(r)]
assert not errors
requests, stages = [], []
for record in records:
    requests.append({k:record[k] for k in ['request_id','release_id','outcome','cache_status','cache_origin_request_id','total_latency_ms','total_cost_microusd']} | record['usage'])
    for position, stage in enumerate(record['stages'],1): stages.append({'request_id':record['request_id'],'outcome':record['outcome'],'position':position} | stage)
request_df, stage_df = pd.DataFrame(requests), pd.DataFrame(stages)
recon = request_df.set_index('request_id')[['total_latency_ms','total_cost_microusd']].join(stage_df.groupby('request_id').agg(stage_latency_ms=('latency_ms','sum'),stage_cost_microusd=('cost_microusd','sum')))
recon['latency_error_ms'] = recon.total_latency_ms-recon.stage_latency_ms
recon['cost_error_microusd'] = recon.total_cost_microusd-recon.stage_cost_microusd
assert len(request_df)==5 and request_df.request_id.is_unique
assert (recon[['latency_error_ms','cost_error_microusd']]==0).all().all()
assert (stage_df[['latency_ms','cost_microusd','input_tokens','output_tokens']]>=0).all().all()
display(recon)
print(f'Verified fixture contract: {fixture_version}')
print('PASS: five schema-valid requests reconcile to 0 ms and 0 micro-USD error.')

**Common Pitfalls:** aggregating before checking missing stages makes corruption look precise; summing overlapping production spans double-counts wall time.

**Quick Health Check:** five unique schema-valid requests; zero latency/cost error; non-negative values. The previous cell runs every check.

> **Checkpoint:** attribution error moves from unknown to exactly zero.

**Reflection bridge:** Reconciliation tells you the ledger is internally consistent, not which stage owns the experience. The next view must preserve the same requests while exposing tails and stage attribution.

## 2 · Replace the Average with Attribution and Tails

**Predict:** Does (1) 201.4 ms describe most users, (2) retrieval own the largest p95, or (3) generation own p50/p95 while cache pulls down the average?

```mermaid
flowchart LR
 A["Gateway"] --> B["Cache"]
 B -->|miss| C["Embedding"] --> D["Retrieval"] --> E["Reranking"] --> F["Generation"]
 F -->|failed| G["Backoff"] --> H["Retry"]
 B -->|hit| I["Return"]
 classDef input fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 classDef process fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 classDef caution fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 classDef success fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 class A input
 class B,C,D,E,F,H process
 class G caution
 class I success
```

In [ ]:
# -- Compute nearest-rank metrics and draw attribution visuals -----------
def nearest_rank(values,q):
    ordered=sorted(float(v) for v in values)
    return ordered[max(0,math.ceil(q*len(ordered))-1)]
stage_order=['gateway','cache_lookup','embedding','retrieval','reranking','generation','retry_backoff']
stage_summary=pd.DataFrame([{'stage_name':n,'observations':len(s),'p50_ms':nearest_rank(s.latency_ms,.5),'p95_ms':nearest_rank(s.latency_ms,.95),'total_cost_microusd':s.cost_microusd.sum()} for n in stage_order if not (s:=stage_df[stage_df.stage_name==n]).empty])
successful=request_df[request_df.outcome=='success']
success_p50,success_p95=nearest_rank(successful.total_latency_ms,.5),nearest_rank(successful.total_latency_ms,.95)
primary=stage_summary.sort_values(['p95_ms','p50_ms'],ascending=False).iloc[0].stage_name
print(f'Prediction result: option 3; {primary} owns the largest p50/p95. Successful p50/p95={success_p50:.0f}/{success_p95:.0f} ms.')

plt.rcParams.update({'figure.facecolor':'#1a1a2e','axes.facecolor':'#1a1a2e','axes.edgecolor':'#e2e8f0','axes.labelcolor':'#e2e8f0','xtick.color':'#e2e8f0','ytick.color':'#e2e8f0','text.color':'#e2e8f0','grid.color':'#475569'})
colors={'gateway':'#38bdf8','cache_lookup':'#22c55e','embedding':'#a78bfa','retrieval':'#f59e0b','reranking':'#fb7185','generation':'#ef4444','retry_backoff':'#94a3b8'}
fig,ax=plt.subplots(figsize=(12,5))
for row,rid in enumerate(request_df.request_id):
    cursor=0
    for st in stage_df[stage_df.request_id==rid].sort_values('position').itertuples(index=False):
        ax.barh(row,st.latency_ms,left=cursor,height=.6,color=colors[st.stage_name],edgecolor='#e2e8f0',hatch='//' if st.status=='failed' else None)
        if st.latency_ms>=20: ax.text(cursor+st.latency_ms/2,row,f'{st.stage_name} a{st.attempt}',ha='center',va='center',fontsize=8)
        cursor+=st.latency_ms
ax.set_yticks(range(5),request_df.request_id); ax.invert_yaxis(); ax.set_xlabel('ms from request start'); ax.set_title('Gantt timeline: retries extend; cache stops early'); ax.grid(axis='x',alpha=.2); plt.tight_layout(); plt.show()
fig,axes=plt.subplots(2,2,figsize=(13,9)); x=np.arange(len(stage_summary))
axes[0,0].bar(request_df.request_id,request_df.total_latency_ms,color=['#22c55e' if x=='success' else '#ef4444' for x in request_df.outcome]); axes[0,0].set_title('Request latency')
axes[0,1].bar(x-.18,stage_summary.p50_ms,.36,label='p50',color='#38bdf8'); axes[0,1].bar(x+.18,stage_summary.p95_ms,.36,label='p95',color='#fb7185'); axes[0,1].set_xticks(x,stage_summary.stage_name,rotation=30,ha='right'); axes[0,1].legend(frameon=False); axes[0,1].set_title('Stage tails')
axes[1,0].bar(stage_summary.stage_name,stage_summary.total_cost_microusd,color='#f59e0b'); axes[1,0].tick_params(axis='x',rotation=30); axes[1,0].set_title('Stage cost')
tokens=request_df[['billed_input_tokens','billed_output_tokens','served_output_tokens']].sum(); axes[1,1].bar(['Billed input','Billed output','Served output'],tokens,color=['#a78bfa','#fb7185','#22c55e']); axes[1,1].set_title('Token flow')
for axis in axes.flat: axis.grid(axis='y',alpha=.2)
plt.tight_layout(); plt.show()

**Common Pitfalls:** total accumulated time confounds frequency with slowness; overlapped spans are not additive. Report count, p50, p95, cost, and critical path together.

**Quick Health Check:** seven stages; counts sum to ledger rows; p95 >= p50; generation p95=200 ms and ranks first.

In [ ]:
# -- Health check: attribution -------------------------------------------
assert set(stage_summary.stage_name)==set(stage_order) and stage_summary.observations.sum()==len(stage_df)
assert (stage_summary.p95_ms>=stage_summary.p50_ms).all() and primary=='generation'
assert stage_summary.set_index('stage_name').loc['generation','p95_ms']==200
print('PASS: generation is the stage bottleneck; retrieval-first tuning is unsupported.')

**Reflection bridge:** Stage p95 names generation as the owner, but total generation latency still mixes waiting, first-token delay, and stream speed. Split those clocks before choosing a serving change.

> **Checkpoint:** the average becomes successful p50/p95 165/285 ms and generation p50/p95 100/200 ms.

## 3 · Separate TTFT, TPOT, and Useful Throughput

Failed attempts do not have a successful stream, so TTFT/TPOT use each request's final successful generation only.

```mermaid
flowchart LR
 A["Start"] --> B["Gateway and RAG"] --> C["Queue and prefill"] --> D["First token: TTFT"] --> E["Later tokens: TPOT"] --> F["Complete"]
 classDef input fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 classDef process fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 classDef success fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 class A input
 class B,C,D,E process
 class F success
```

**Your turn:** switch `successful_only`; both populations have explicit expected results.

In [ ]:
# -- Your turn plus TTFT, TPOT, and throughput checks --------------------
successful_only=True  # CHANGE THIS: False includes the paid failure
population=successful if successful_only else request_df
assert nearest_rank(population.total_latency_ms,.5)==(165 if successful_only else 265)
final_gen=stage_df.query("stage_name=='generation' and status=='success' and outcome=='success'").sort_values(['request_id','attempt']).groupby('request_id',as_index=False).tail(1)
ttft_p50,ttft_p95=nearest_rank(final_gen.ttft_ms,.5),nearest_rank(final_gen.ttft_ms,.95)
tpot_p50,tpot_p95=nearest_rank(final_gen.tpot_ms,.5),nearest_rank(final_gen.tpot_ms,.95)
throughput=final_gen.output_tokens.sum()/(final_gen.latency_ms.sum()/1000)
assert len(final_gen)==3 and (ttft_p50,ttft_p95,tpot_p50,tpot_p95)==(50,60,15,20)
assert math.isclose(throughput,57.14285714285714)
print(f'PASS: TTFT={ttft_p50:.0f}/{ttft_p95:.0f}, TPOT={tpot_p50:.0f}/{tpot_p95:.0f}, serial output rate={throughput:.2f} tokens/s.')
print('This is not concurrent capacity; queueing and batching are absent.')

**Common Pitfalls:** unnamed percentile interpolation, failed attempts in streaming metrics, and serial tokens/s called server throughput. Name convention, population, and workload.

**Quick Health Check:** 4 successful requests; 3 final successful generations; TTFT 50/60; TPOT 15/20; 24 tokens over 420 ms.

> **Checkpoint:** first-token delay, streaming speed, and serial useful work are now separate.

**Reflection bridge:** TTFT and TPOT explain the successful stream, but retries and cache hits change how often that work happens and who pays for failed attempts. User experience and economics now need the same denominator discipline.

## 4 · Expose Retries, Cache, Tokens, and Cost

**Predict:** Six attempts across four generation-bearing requests is 1.50. Across three successful uncached requests, four attempts is 1.33. Both are correct only with named denominators.

```mermaid
flowchart LR
 A["Cache"] -->|hit| B["Serve origin"]
 A -->|miss| C["Attempt 1"] -->|failure| D["Backoff"] --> E["Attempt 2"]
 E -->|failure| F["Paid failure"]
 C -->|success| G["Served output"]
 E -->|success| G
 classDef input fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 classDef process fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 classDef failure fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 classDef success fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 class A input
 class C,D,E process
 class F failure
 class B,G success
```

In [ ]:
# -- Measure retries, cache, tokens, cost, and retry timeline ------------
gen=stage_df[stage_df.stage_name=='generation']; all_amp=len(gen)/gen.request_id.nunique()
success_uncached=set(request_df.query("outcome=='success' and cache_status=='miss'").request_id)
success_attempts=gen[gen.request_id.isin(success_uncached)]; success_amp=len(success_attempts)/len(success_uncached)
failed_attempt_cost=int(gen[gen.status=='failed'].cost_microusd.sum())
hit=request_df[request_df.cache_status=='exact_hit'].iloc[0]; origin=request_df.set_index('request_id').loc[hit.cache_origin_request_id]
saved_ms=int(origin.total_latency_ms-hit.total_latency_ms); saved_cost=int(origin.total_cost_microusd-hit.total_cost_microusd)
success_count=int((request_df.outcome=='success').sum()); total_cost=int(request_df.total_cost_microusd.sum()); cost_per_success=total_cost/success_count
billed_in=int(request_df.billed_input_tokens.sum()); billed_out=int(request_df.billed_output_tokens.sum()); served_out=int(request_df.served_output_tokens.sum())
assert (all_amp,round(success_amp,6),saved_ms,saved_cost)==(1.5,round(4/3,6),158,1030)
assert (billed_in,billed_out,served_out,total_cost,cost_per_success)==(600,36,30,6420,1605)
print(f'Retries: {all_amp:.2f} all generation-bearing; {success_amp:.2f} successful uncached; failed attempts cost {failed_attempt_cost:,} micro-USD.')
print(f'Cache pair saves {saved_ms} ms and {saved_cost:,} micro-USD. Cost per success={cost_per_success:,.0f} micro-USD.')
fig,ax=plt.subplots(figsize=(10,3.4))
for row,rid in enumerate(['req-003','req-005']):
    cursor=0
    for st in stage_df[(stage_df.request_id==rid)&stage_df.stage_name.isin(['generation','retry_backoff'])].sort_values('position').itertuples(index=False):
        color='#94a3b8' if st.stage_name=='retry_backoff' else ('#22c55e' if st.status=='success' else '#ef4444')
        ax.barh(row,st.latency_ms,left=cursor,height=.55,color=color,edgecolor='#e2e8f0'); ax.text(cursor+st.latency_ms/2,row,'backoff' if st.stage_name=='retry_backoff' else f'a{st.attempt} {st.status}',ha='center',va='center'); cursor+=st.latency_ms
ax.set_yticks([0,1],['req-003','req-005']); ax.invert_yaxis(); ax.set_xlabel('generation-path ms'); ax.set_title('Retry timeline: recovery and terminal failure consume budget'); plt.tight_layout(); plt.show()

**Your turn:** change `retry_population` below.

**Common Pitfalls:** unnamed retry denominators, hypothetical cache hit rates, successful-row-only cost, and billed output treated as served output. Keep failed work and label every population.

**Quick Health Check:** 6/4 and 4/3 attempts; valid cache origin; zero hit billing/generation; 600/36/30 token counters; 6,420 total and 1,605 micro-USD/success.

In [ ]:
# -- Your turn and health check: denominator and cache -------------------
retry_population='generation-bearing'  # CHANGE THIS: 'successful-uncached'
reported=all_amp if retry_population=='generation-bearing' else success_amp
assert math.isclose(reported,{'generation-bearing':1.5,'successful-uncached':4/3}[retry_population])
hit_stages=set(stage_df[stage_df.request_id==hit.request_id].stage_name)
assert hit.billed_input_tokens==0 and hit.billed_output_tokens==0 and 'generation' not in hit_stages
assert hit.served_output_tokens==origin.served_output_tokens
print(f'PASS: {retry_population} amplification={reported:.2f}; cache and economics reconcile.')

**Reflection bridge:** The ledger now exposes retry amplification, cache savings, and useful-work cost. Those measurements still do not authorize a configuration change; they only rank the next discriminating test.

> **Checkpoint:** retries, cache, billed/served tokens, and failed spend are visible.

## 5 · Make a Bottleneck-First Decision

```mermaid
flowchart TD
 A["Reconciled stage p95"] --> B{"Highest stage?"}
 B -->|generation| C["Split queue, prefill, decode"]
 A --> D{"Attempts > requests?"}
 D -->|Yes| E["Audit retryability and deadlines"]
 C --> F["Change one variable only after test"]
 E --> F
 classDef input fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 classDef process fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 classDef caution fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 classDef success fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 class A input
 class B,D caution
 class C,E process
 class F success
```

**Predict:** The supported first move is not retrieval tuning or more retries. It is to split generation into queue/prefill/decode and audit retry eligibility.

In [ ]:
# -- Rank bottleneck and name the next discriminating test --------------
decision=stage_summary.copy(); decision['cost_share_pct']=decision.total_cost_microusd/total_cost*100
decision=decision.sort_values(['p95_ms','cost_share_pct'],ascending=False).reset_index(drop=True)
tests={'generation':'Separate queue, prefill, decode; audit retryable failures','retrieval':'Split client, network, filter, index','reranking':'Measure candidate count and batch','embedding':'Measure batching and network','gateway':'Measure policy/network','cache_lookup':'Measure key/store','retry_backoff':'Check deadline and jitter'}
decision['next_test']=decision.stage_name.map(tests)
selected_stage,selected_action=decision.iloc[0].stage_name,decision.iloc[0].next_test
generation_share=float(decision.loc[decision.stage_name=='generation','cost_share_pct'].iloc[0])
assert selected_stage=='generation' and generation_share>95 and all_amp>1
display(decision.round({'cost_share_pct':1}))
print(f'PASS: measure {selected_stage} first: {selected_action}. No configuration change approved.')

**Common Pitfalls:** tuning what your team owns, treating generation as one black box, or raising retries after one recovery. Rank measured tails/cost, split one level deeper, and name a falsifying test.

**Quick Health Check:** generation first by p95, over 95% of cost, amplification >1, and next action is measurement rather than configuration.

> **Checkpoint:** profile queue/prefill/decode and audit retries; do not optimize retrieval or raise retry count.

**Reflection bridge:** A bottleneck decision is only as safe as the telemetry that carries it. High-cardinality or content-bearing labels could make a technically correct dashboard operationally unsafe.

## 6 · Content-Free Telemetry and Production Limits

```mermaid
flowchart LR
 A["Trace context"] --> B["Spans and measurements"]
 C["Bounded attributes"] --> B --> D["Aggregate metrics"]
 E["Content and high-cardinality IDs"] --> F["Forbidden labels"]
 classDef input fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 classDef process fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 classDef failure fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 classDef success fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 class A,C input
 class B process
 class D success
 class E,F failure
```

Request IDs belong in trace context, never metric dimensions. Exact tokens/latency/cost are measurements; labels use bounded buckets. Prompts, completions, user/tenant/document/source identifiers are denied by default.

**Common Pitfalls:** IDs or exact token counts as labels create cardinality/privacy risk; default body and exception logging can leak content.

**Exact production health checks:** calibrate load engines; pin release/workload/token/cache/retry versions; record offered/achieved load and queueing; report p50/p95/p99 total/TTFT/TPOT; run warm, step, peak, burst, soak, dependency failure, instance loss, recovery, and blue/green overlap; stop at first SLO/safety breach; reconcile billing; inspect exported telemetry for body/header/exception/environment leakage; retain raw and normalized evidence.

In [ ]:
# -- Health check: bounded telemetry and evidence boundary ---------------
telemetry=json.loads((ROOT/'projects/riverside-ai-platform/contracts/v1/telemetry-attributes.schema.json').read_text(encoding='utf-8'))
keys,required=set(telemetry['properties']),set(telemetry['required'])
forbidden={'request_id','trace_id','user_id','tenant_id','prompt','completion','document_id','chunk_id','source_uri'}
assert telemetry['additionalProperties'] is False and required<=keys and forbidden.isdisjoint(keys)
assert {'model.release_id','cache.result','gen_ai.usage.prompt_tokens_bucket','gen_ai.usage.output_tokens_bucket'}<=keys
evidence_state='LOCAL_FIXTURE'
print(f'PASS: {len(keys)} bounded attributes; evidence remains {evidence_state}.')
print('UNVALIDATED: production SLO, capacity, billing, exporter/redaction, cardinality, and Azure behavior.')

**Reflection bridge:** Bounded telemetry protects the measurement path, but local fixture arithmetic still needs a retained report and an explicit handoff that refuses production claims.

## 7 · Package the Evidence and Hand Off

```mermaid
flowchart LR
 A["Fixture digest"] --> B["Metrics and health checks"] --> C["Bottleneck decision"] --> D["Local report"] --> E["Production revalidation"]
 classDef input fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 classDef process fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 classDef success fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 class A input
 class B,C,D process
 class E success
```

In [ ]:
# -- Final frozen-outcome checks and report ------------------------------
actual={'success_rate':success_count/len(request_df),'success_p50_ms':success_p50,'success_p95_ms':success_p95,'ttft_p50_ms':ttft_p50,'ttft_p95_ms':ttft_p95,'tpot_p50_ms':tpot_p50,'tpot_p95_ms':tpot_p95,'throughput':throughput,'retry_all':all_amp,'retry_success':success_amp,'cache_latency_saved_ms':saved_ms,'cache_cost_saved_microusd':saved_cost,'total_cost_microusd':total_cost,'cost_per_success_microusd':cost_per_success}
expected={'success_rate':.8,'success_p50_ms':165,'success_p95_ms':285,'ttft_p50_ms':50,'ttft_p95_ms':60,'tpot_p50_ms':15,'tpot_p95_ms':20,'throughput':57.14285714285714,'retry_all':1.5,'retry_success':4/3,'cache_latency_saved_ms':158,'cache_cost_saved_microusd':1030,'total_cost_microusd':6420,'cost_per_success_microusd':1605}
assert all(math.isclose(actual[k],v) for k,v in expected.items())
digest=hashlib.sha256(TRACE.read_bytes()).hexdigest()
report={'schema_version':'ai-eng.latency-cost-report.v1','evidence_state':evidence_state,'generated_at_utc':datetime.now(timezone.utc).isoformat(),'release_id':request_df.release_id.unique().item(),'source':{'path':TRACE.relative_to(ROOT).as_posix(),'sha256':digest,'request_count':len(request_df)},'health':{'schema_valid':not errors,'latency_reconciled':True,'cost_reconciled':True,'bounded_metrics_checked':True},'metrics':actual|{'billed_input_tokens':billed_in,'billed_output_tokens':billed_out,'served_output_tokens':served_out},'decision':{'primary_stage':selected_stage,'next_test':selected_action,'configuration_change_approved':False},'limitations':['production SLO not measured','capacity not measured','billing not reconciled','Azure not validated']}
out=CHAPTER/'artifacts'; out.mkdir(parents=True,exist_ok=True); path=out/'latency-cost-report.json'; path.write_text(json.dumps(report,indent=2,sort_keys=True)+'\n',encoding='utf-8')
print(f'PASS: frozen outcomes match; saved {path.relative_to(ROOT)}. Evidence remains LOCAL_FIXTURE.')

### Three-Tier Coverage

| Tier | Techniques |
|---|---|
| Built and measured | Conservation, nearest-rank p50/p95, Gantt/multi-panel attribution, TTFT/TPOT, output rate, retries, cache, billed/served tokens, cost/success, decision scorecard, telemetry allowlist, report |
| Explained and illustrated | Parallel critical path, queue/prefill/decode, deadlines, bounded metrics, load stages |
| Named with reason | p99/confidence need volume; batching/KV pressure need a scheduler; autoscaling needs deployment state; billing needs exports; semantic-cache safety needs evaluation |

### Completed Roadmap

| Step | Failure exposed | What the notebook proves when run |
|---:|---|---|
| 1 | Averages were trusted before ledger checks | Five requests reconcile to zero latency and cost error |
| 2 | Total latency had no owner | Generation owns the largest measured stage p50/p95 |
| 3 | Generation latency hid user-visible clocks | TTFT, TPOT, and serial useful throughput are separated |
| 4 | Retries and cache distorted spend | Amplification, failed spend, observed cache savings, and cost per success reconcile |
| 5 | Optimization began from intuition | Generation phases and retry eligibility become the next falsifying tests |
| 6 | Telemetry could leak content or explode cardinality | The bounded attribute contract excludes content and request identity |
| 7 | Local measurements invited production overclaims | One report retains fixture metrics and leaves capacity, billing, and Azure unvalidated |

### Key Takeaways

- Reconcile before aggregating.
- Name percentile convention and population.
- TTFT and TPOT demand different tests.
- Failed and retried work stays in successful-work economics.
- Cache savings need an observed origin; production hit rate needs production evidence.
- Split the measured bottleneck one level deeper before changing one variable.
- Keep correlation in traces and bounded dimensions on metrics.

> **Final checkpoint:** the strongest supported decision is `measure generation phases and retry policy next`. No production SLO, capacity, provider bill, or Azure validation is claimed.